# Python Decorators — Advanced Patterns & Algorithm Profiling

> **Topic:** Python Decorators | **Folder:** Data Structures & Algorithms

A **decorator** is a design pattern in Python that allows you to dynamically modify or extend the
behavior of a function or class **without changing its source code**.

```python
@my_decorator
def my_function():
    pass

# Equivalent to: my_function = my_decorator(my_function)
```

In Data Structures & Algorithms, decorators are indispensable for **profiling execution time**,
**tracking recursive call depth**, **memoizing dynamic programming solutions**, and **validating inputs**.

---

## Table of Contents
1. [First-Class Functions & Closures Foundation](#1.-First-Class-Functions-&-Closures-Foundation)
2. [The Decorator Syntax (`@`)](#2.-The-Decorator-Syntax-(@))
3. [Handling Function Arguments (`*args`, `**kwargs`)](#3.-Handling-Function-Arguments-(*args,-**kwargs))
4. [Preserving Function Metadata (`functools.wraps`)](#4.-Preserving-Function-Metadata-(functools.wraps))
5. [Algorithmic Decorators: Timing, Call Counting & Trace](#5.-Algorithmic-Decorators:-Timing,-Call-Counting-&-Trace)
6. [Memoization Decorator (Dynamic Programming Optimization)](#6.-Memoization-Decorator-(Dynamic-Programming-Optimization))
7. [Decorator Factories (Decorators with Arguments)](#7.-Decorator-Factories-(Decorators-with-Arguments))
8. [Stacking Multiple Decorators](#8.-Stacking-Multiple-Decorators)
9. [Class-Based Decorators (`__call__`)](#9.-Class-Based-Decorators-(__call__))
10. [Input Validation & Type Checking Decorators](#10.-Input-Validation-&-Type-Checking-Decorators)
11. [Quick Reference Card](#11.-Quick-Reference-Card)


---
## 1. First-Class Functions & Closures Foundation

To understand decorators, remember three key Python behaviors:
1. **Functions are First-Class Objects**: They can be assigned to variables, passed as arguments, and returned from other functions.
2. **Functions can be Nested**: A function can be defined inside another function.
3. **Closures**: An inner function retains access to variables in its enclosing scope even after the outer function finishes executing.


In [ ]:
# 1. Passing a function as an argument
def apply_operation(func, a, b):
    return func(a, b)

def add(x, y): return x + y
def multiply(x, y): return x * y

print(f"apply_operation(add, 5, 3)      = {apply_operation(add, 5, 3)}")
print(f"apply_operation(multiply, 5, 3) = {apply_operation(multiply, 5, 3)}")


In [ ]:
# 2 & 3. Nested function returning a closure
def make_logger(prefix):
    def log_message(msg):          # Inner function closes over 'prefix'
        print(f"[{prefix.upper()}] {msg}")
    return log_message             # Return the inner function object

info_logger = make_logger("info")
error_logger = make_logger("error")

info_logger("Algorithm initialized successfully.")
error_logger("Index out of bounds exception detected!")


---
## 2. The Decorator Syntax (`@`)

A decorator is simply a function that takes another function as an argument,
wraps it with additional logic, and returns the modified wrapper function.


In [ ]:
# Step 1: Define a basic decorator
def my_simple_decorator(func):
    def wrapper():
        print("==> [BEFORE] Preparing to execute algorithm...")
        func()
        print("<== [AFTER] Algorithm execution finished.\n")
    return wrapper

# Step 2: Decorating manually
def run_search():
    print("    Executing binary search on array...")

decorated_search = my_simple_decorator(run_search)
decorated_search()


In [ ]:
# Step 3: Using the clean @ syntax
@my_simple_decorator
def run_sort():
    print("    Executing quicksort on array...")

# Invoking run_sort() automatically invokes the wrapped version!
run_sort()


---
## 3. Handling Function Arguments (`*args`, `**kwargs`)

To allow decorators to wrap functions with **any parameters and return values**,
the inner `wrapper` function must accept `*args` and `**kwargs`, and explicitly return
the result of calling the target function.


In [ ]:
# Universal Decorator Pattern
def audit_call(func):
    def wrapper(*args, **kwargs):
        print(f"[AUDIT] Calling '{func.__name__}' with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)   # Capture return value
        print(f"[AUDIT] '{func.__name__}' returned -> {result}")
        return result                    # Return value to caller
    return wrapper

@audit_call
def compute_pow(base, exponent, modulus=None):
    if modulus:
        return pow(base, exponent, modulus)
    return base ** exponent

val1 = compute_pow(2, 10)
val2 = compute_pow(2, 10, modulus=1000)


---
## 4. Preserving Function Metadata (`functools.wraps`)

When a function is decorated, its identity (`__name__`, `__doc__`, `__annotations__`)
gets overwritten by the wrapper function.  
Use `@functools.wraps(func)` inside the decorator to preserve original metadata!


In [ ]:
from functools import wraps

# Without @wraps
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def target_one():
    """This is target one docstring."""
    pass

print("Without @wraps:")
print(f"  Name: {target_one.__name__}")   # Prints 'wrapper'
print(f"  Doc : {target_one.__doc__}")    # Prints None

# With @wraps
def good_decorator(func):
    @wraps(func)                          # Preserves identity!
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def target_two():
    """This is target two docstring."""
    pass

print("\nWith @wraps:")
print(f"  Name: {target_two.__name__}")   # Prints 'target_two'
print(f"  Doc : {target_two.__doc__}")    # Prints docstring


---
## 5. Algorithmic Decorators: Timing, Call Counting & Trace

Decorators are ideal for profiling algorithms without cluttering core logic.


In [ ]:
# 1. Timer Decorator for Execution Time Benchmarking
import time
from functools import wraps

def timeit(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱️ [{func.__name__}] Execution time: {elapsed:.6f} seconds")
        return result
    return wrapper

@timeit
def bubble_sort(arr):
    arr = list(arr)
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr

sample_data = list(range(1000, 0, -1))
sorted_data = bubble_sort(sample_data)


In [ ]:
# 2. Call Counter Decorator to Measure Recursion Overhead
def count_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0   # Attach call counter attribute
    return wrapper

@count_calls
def fib_naive(n):
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

res = fib_naive(15)
print(f"fib_naive(15) = {res}")
print(f"Total recursive calls made: {fib_naive.calls}")


---
## 6. Memoization Decorator (Dynamic Programming Optimization)

Memoization caches function call results based on input parameters,
reducing exponential $O(2^n)$ recursive algorithms to linear $O(n)$ time complexity!


In [ ]:
# Building an LRU / Memoize Decorator from scratch
def memoize(func):
    cache = {}
    @wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    wrapper.cache = cache
    return wrapper

@count_calls
@memoize
def fib_memoized(n):
    if n <= 1:
        return n
    return fib_memoized(n - 1) + fib_memoized(n - 2)

fib_memoized.calls = 0
res_memo = fib_memoized(35)
print(f"fib_memoized(35) = {res_memo}")
print(f"Total calls with memoization: {fib_memoized.calls} (vs ~29 million for naive!)")


---
## 7. Decorator Factories (Decorators with Arguments)

When a decorator requires custom options (e.g. `@retry(max_attempts=3)`),
you write an extra outer function (**decorator factory**) that returns the actual decorator.

```python
def decorator_factory(option):
    def actual_decorator(func):
        def wrapper(*args, **kwargs):
            # option and func are available here
            return func(*args, **kwargs)
        return wrapper
    return actual_decorator
```


In [ ]:
# Retry Decorator Factory
import random
random.seed(42)

def retry(max_attempts=3, delay=0.1):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"  ⚠️ Attempt {attempt}/{max_attempts} failed: {e}")
                    if attempt == max_attempts:
                        print("  ❌ Max retries reached. Raising error.")
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

@retry(max_attempts=3, delay=0.01)
def unstable_network_call():
    if random.random() < 0.7:
        raise ConnectionError("Network timeout")
    return "🌐 Data fetched successfully!"

try:
    print(unstable_network_call())
except ConnectionError:
    print("Handled final failure gracefully.")


---
## 8. Stacking Multiple Decorators

Multiple decorators can be applied to a single function.
They are executed in **bottom-up order** (innermost decorator wraps first).

```python
@dec1
@dec2
def func(): pass

# Equivalent to: func = dec1(dec2(func))
```


In [ ]:
def make_bold(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def make_italic(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper

# Applied bottom-up: make_italic runs first, then make_bold
@make_bold
@make_italic
def render_title(text):
    return text

print(render_title("Data Structures & Algorithms"))


---
## 9. Class-Based Decorators (`__call__`)

A class implementing the `__call__` dunder method can act as a decorator.
Class decorators are great when you need to maintain state or complex configurations.


In [ ]:
class CallStatistics:
    def __init__(self, func):
        self.func = func
        self.calls = 0
        self.total_time = 0.0
        wraps(func)(self)

    def __call__(self, *args, **kwargs):
        self.calls += 1
        start = time.perf_counter()
        result = self.func(*args, **kwargs)
        self.total_time += (time.perf_counter() - start)
        return result

    def stats(self):
        avg_time = self.total_time / self.calls if self.calls > 0 else 0
        return f"Stats for '{self.func.__name__}': Calls={self.calls}, Total Time={self.total_time:.6f}s, Avg={avg_time:.6f}s"

@CallStatistics
def linear_search(arr, target):
    for i, x in enumerate(arr):
        if x == target: return i
    return -1

data = list(range(10000))
linear_search(data, 5000)
linear_search(data, 9999)
linear_search(data, -1)

print(linear_search.stats())


---
## 10. Input Validation & Type Checking Decorators

Decorators can enforce preconditions, argument types, or numerical constraints
before an algorithm runs.


In [ ]:
def enforce_types(*types):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for arg, expected_type in zip(args, types):
                if not isinstance(arg, expected_type):
                    raise TypeError(f"Argument '{arg}' is not of expected type {expected_type.__name__}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@enforce_types(int, list)
def binary_search(target, sorted_arr):
    low, high = 0, len(sorted_arr) - 1
    while low <= high:
        mid = (low + high) // 2
        if sorted_arr[mid] == target: return mid
        elif sorted_arr[mid] < target: low = mid + 1
        else: high = mid - 1
    return -1

# Valid call
print("Binary Search index:", binary_search(42, [10, 20, 30, 42, 50]))

# Invalid call raises TypeError
try:
    binary_search("not_an_int", [10, 20])
except TypeError as e:
    print(f"Caught Type Error: {e}")


---
## 11. Quick Reference Card


In [ ]:
# ==================================================================
# PYTHON DECORATORS – QUICK REFERENCE
# ==================================================================
from functools import wraps, lru_cache

# --- Basic Decorator Pattern ---
def basic_dec(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # Pre-execution logic
        res = func(*args, **kwargs)
        # Post-execution logic
        return res
    return wrapper

# --- Factory Decorator (With Arguments) ---
def param_dec(option="default"):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        return wrapper
    return decorator

# --- Built-in LRU Cache Decorator ---
@lru_cache(maxsize=128)
def fib(n):
    return n if n < 2 else fib(n-1) + fib(n-2)

print("fib(20) =", fib(20))


---
## Summary

| Decorator Pattern | Use Case | Key Function / Syntax |
|-------------------|----------|------------------------|
| **Basic Wrapper** | Add logging/timing to function | `def dec(f): def wrap(*a, **kw): ...` |
| **Metadata Preserver** | Keep function name & docstrings | `@functools.wraps(func)` |
| **Memoization** | Optimize recursive algorithms | `@functools.lru_cache` or custom dict cache |
| **Decorator Factory** | Pass arguments to decorator | Extra enclosing function returning decorator |
| **Stacked Decorators** | Apply multiple modifiers | Evaluated bottom-up: `@dec1 
 @dec2` |
| **Class Decorator** | Maintain state across calls | Implement `__call__(self, *args)` |
| **Type / Guard** | Precondition verification | Raise `TypeError`/`ValueError` inside wrapper |

---
*Next up: **Generators & Iterators in Data Structures***
